# Prompt Engineering Best Practices: Iterative Prompt Development

[Article](https://todatabeyond.com/blog/prompt-engineering-best-practices-iterative-prompt-development). [Series contents](./README.md).

Install `pip install openai==2.43.0 python-dotenv==1.2.2 bleach==6.4.0 beautifulsoup4==4.14.3`. Set OPENAI_API_KEY in your environment (or an uncommitted .env); API calls incur charges. MODEL defaults to the tested gpt-5.4-mini. Never place credentials in cells. The examples request ordinary task outputs; responses can change between runs. The nonexecuting appendix preserves original source code and transcripts.

In [ ]:
import os, json, re, sys
from importlib.metadata import version
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()
MODEL = os.getenv("MODEL", "gpt-5.4-mini")
client = OpenAI(max_retries=0, timeout=60)
def get_completion(prompt, model=MODEL):
    result = client.chat.completions.create(
        model=model, messages=[{"role": "user", "content": prompt}],
        reasoning_effort="none", max_completion_tokens=2048)
    if result.choices[0].finish_reason != "stop":
        raise RuntimeError("The completion did not finish normally")
    content = result.choices[0].message.content
    if not content:
        raise RuntimeError("No text returned")
    return content
def parse_json_object(text):
    decoder = json.JSONDecoder()
    for index, char in enumerate(text):
        if char == "{":
            try:
                value, _ = decoder.raw_decode(text[index:])
            except json.JSONDecodeError:
                continue
            if isinstance(value, dict):
                return value
    raise ValueError("No JSON object found")
print("Python:", sys.version.split()[0])
print("openai:", version("openai"), "python-dotenv:", version("python-dotenv"))
print("Model:", MODEL)

In [ ]:
fact_sheet_laptop = """
OVERVIEW

Introducing our sleek Laptop Series, an embodiment of cutting-edge design and functionality for modern workspaces.
Part of a sophisticated family of tech-inspired devices, including laptops, docking stations, accessories, and more.
Available in multiple colors and finishes to seamlessly integrate with your personal or professional aesthetic.
Crafted with premium materials for a durable and stylish appeal.
Suitable for a variety of environments, whether it's your home office or a corporate setting.
Engineered for superior performance and productivity, making it an ideal choice for professionals.
DESIGN

The laptop is available in various color options, allowing you to personalize it to your preference.
Choose between different finishes, including brushed aluminum, matte black, glossy white, or chrome accents.
Select from configurations with a standard display or a touchscreen option.
Options for additional features such as a backlit keyboard, fingerprint sensor, or face recognition technology.
Sleek and lightweight design for portability without compromising on power.
SPECIFICATIONS

Powerful Intel/AMD processors for seamless multitasking and high-performance computing.
Ultra-thin design with dimensions:
WIDTH 35 CM | 13.78”
DEPTH 24 CM | 9.45”
HEIGHT 1.5 CM | 0.59”
Crisp and vibrant display with a resolution tailored for immersive work or entertainment experiences.
OPTIONS

Choose between SSD or HDD storage options for speed and capacity tailored to your needs.
Various connectivity options, including USB-C, HDMI, and Thunderbolt for versatile compatibility.
Different RAM configurations to suit your processing requirements.
Options for operating systems, catering to both Windows and MacOS preferences.
MATERIALS

Durable aluminum casing for a premium look and robust protection.
High-resolution display with anti-glare coating for comfortable viewing in any lighting condition.
Cutting-edge cooling technology to ensure optimal performance during extended usage.
COUNTRY OF ORIGIN

Crafted with precision in state-of-the-art facilities in Japan, combining traditional craftsmanship with technological innovation.

"""
print("Laptop fact sheet ready:", len(fact_sheet_laptop.split()), "words")

In [ ]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
initial_response = response

In [ ]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Use at most 50 words.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
short_response = response

In [ ]:
word_count = len(short_response.split())
print("Word count:", word_count)

In [ ]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for laptop retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.


Use at most 50 words.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
focused_response = response

In [ ]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for laptop retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

Use at most 50 words.

Technical specifications: ```{fact_sheet_laptop}```
"""
prompt += "\nOnly include Product IDs explicitly present in the fact sheet. If none are present, write Product IDs: not provided. Do not invent identifiers."
response = get_completion(prompt)
print(response)
id_response = response

In [ ]:
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for laptop retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

After the description, include a table that gives the 
product's dimensions. The table should have two columns.
In the first column include the name of the dimension. 
In the second column include the measurements in inches only.

Give the table the title 'Product Dimensions'.

Format everything as HTML that can be used in a website. 
Place the description in a <div> element.

Technical specifications: ```{fact_sheet_laptop}```
"""

prompt += "\nOnly include Product IDs explicitly present in the fact sheet. If none are present, write Product IDs: not provided. Do not invent identifiers."
response = get_completion(prompt)
print(response)
html_response = response

In [ ]:
import bleach
from bs4 import BeautifulSoup
from IPython.display import display, HTML
allowed_tags = ["div", "h2", "h3", "p", "table", "caption", "thead", "tbody", "tr", "th", "td", "strong", "em", "br"]
safe_html = bleach.clean(html_response, tags=allowed_tags, attributes={}, strip=True)
document = BeautifulSoup(safe_html, "html.parser")
assert document.select_one("table") is not None
assert len(document.select("table tr")) == 4
print(safe_html)
display(HTML(safe_html))

In [ ]:
assert word_count == len(short_response.split())
assert 0 < word_count <= 50
assert all(fake not in id_response for fake in ["Laptop01", "Laptop02"])
assert "not provided" in id_response.lower() or "none" in id_response.lower()
rows = document.select("table tr")
dimensions = {row.select_one("td").get_text(strip=True).lower(): row.select("td")[1].get_text(strip=True) for row in rows if len(row.select("td")) == 2}
assert set(dimensions) == {"width", "depth", "height"}
for name, value in {"width": "13.78", "depth": "9.45", "height": "0.59"}.items():
    assert value in dimensions[name]
assert not document.select("script, iframe, img, a")
assert not any(tag.attrs for tag in document.find_all())
probe = bleach.clean('<img src=x onerror=alert(1)><script>alert(1)</script><table onclick="bad()"><tr><td>13.78</td></tr></table>', tags=allowed_tags, attributes={}, strip=True)
parsed = BeautifulSoup(probe, "html.parser")
assert not parsed.select("img, script") and not any(t.attrs for t in parsed.find_all())
assert parse_json_object('```json\n{"ok": true}\n```') == {"ok": True}
print("PASS: measured word limit, no invented source IDs, all three inch dimensions, rendered HTML structure and active-content sanitization.")

## Original source appendix — not executed

Historical SDK calls and transcripts below are retained verbatim for comparison, not used as fresh test results.

### Original pre block 1

```text
import openai
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

openai.api_key  = os.getenv('OPENAI_API_KEY')


def get_completion(prompt, model="gpt-3.5-turbo"):
    messages = [{"role": "user", "content": prompt}]
    response = openai.ChatCompletion.create(
        model=model,
        messages=messages,
        temperature=0, # this is the degree of randomness of the model's output
    )
    return response.choices[0].message["content"]
```

### Original pre block 2

```text
fact_sheet_laptop = """
OVERVIEW

Introducing our sleek Laptop Series, an embodiment of cutting-edge design and functionality for modern workspaces.
Part of a sophisticated family of tech-inspired devices, including laptops, docking stations, accessories, and more.
Available in multiple colors and finishes to seamlessly integrate with your personal or professional aesthetic.
Crafted with premium materials for a durable and stylish appeal.
Suitable for a variety of environments, whether it's your home office or a corporate setting.
Engineered for superior performance and productivity, making it an ideal choice for professionals.
DESIGN

The laptop is available in various color options, allowing you to personalize it to your preference.
Choose between different finishes, including brushed aluminum, matte black, glossy white, or chrome accents.
Select from configurations with a standard display or a touchscreen option.
Options for additional features such as a backlit keyboard, fingerprint sensor, or face recognition technology.
Sleek and lightweight design for portability without compromising on power.
SPECIFICATIONS

Powerful Intel/AMD processors for seamless multitasking and high-performance computing.
Ultra-thin design with dimensions:
WIDTH 35 CM | 13.78”
DEPTH 24 CM | 9.45”
HEIGHT 1.5 CM | 0.59”
Crisp and vibrant display with a resolution tailored for immersive work or entertainment experiences.
OPTIONS

Choose between SSD or HDD storage options for speed and capacity tailored to your needs.
Various connectivity options, including USB-C, HDMI, and Thunderbolt for versatile compatibility.
Different RAM configurations to suit your processing requirements.
Options for operating systems, catering to both Windows and MacOS preferences.
MATERIALS

Durable aluminum casing for a premium look and robust protection.
High-resolution display with anti-glare coating for comfortable viewing in any lighting condition.
Cutting-edge cooling technology to ensure optimal performance during extended usage.
COUNTRY OF ORIGIN

Crafted with precision in state-of-the-art facilities in Japan, combining traditional craftsmanship with technological innovation.

"""
```

### Original pre block 3

```text
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 4

```text
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

Use at most 50 words.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 5

```text
len(response.split())
```

### Original pre block 6

```text
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for laptop retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

Use at most 50 words.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 7

```text
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for laptop retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

Use at most 50 words.

Technical specifications: ```{fact_sheet_laptop}```
"""
response = get_completion(prompt)
print(response)
```

### Original pre block 8

```text
prompt = f"""
Your task is to help a marketing team create a 
description for a retail website of a product based 
on a technical fact sheet.

Write a product description based on the information 
provided in the technical specifications delimited by 
triple backticks.

The description is intended for laptop retailers, 
so should be technical in nature and focus on the 
materials the product is constructed from.

At the end of the description, include every 7-character 
Product ID in the technical specification.

After the description, include a table that gives the 
product's dimensions. The table should have two columns.
In the first column include the name of the dimension. 
In the second column include the measurements in inches only.

Give the table the title 'Product Dimensions'.

Format everything as HTML that can be used in a website. 
Place the description in a <div> element.

Technical specifications: ```{fact_sheet_laptop}```
"""

response = get_completion(prompt)
print(response)
```

### Original pre block 9

```text
from IPython.display import display, HTML
display(HTML(response))
```

### Original quote/transcript 1

```text
This too is why I personally have not paid as much attention to the internet articles that say 30 perfect prompts, because I think, there probably isn’t a perfect prompt for everything under the sun. It’s more important that you have a process for developing a good prompt for your specific application.
```

### Original quote/transcript 2

```text
Introducing our sleek Laptop Series, a perfect blend of cutting-edge design and functionality for modern workspaces. This laptop is part of a sophisticated family of tech-inspired devices, including docking stations, accessories, and more. With its multiple color options and finishes, it seamlessly integrates with your personal or professional aesthetic.
```

### Original quote/transcript 3

```text
Crafted with premium materials, this laptop offers a durable and stylish appeal. Whether you are working from your home office or a corporate setting, this laptop is suitable for a variety of environments. Engineered for superior performance and productivity, it is the ideal choice for professionals.
```

### Original quote/transcript 4

```text
The laptop is available in various color options, allowing you to personalize it to your preference. Choose between brushed aluminum, matte black, glossy white, or chrome accents for different finishes. You also have the option to select a configuration with a standard display or a touchscreen.
```

### Original quote/transcript 5

```text
To enhance your experience, this laptop offers additional features such as a backlit keyboard, fingerprint sensor, or face recognition technology. Despite its sleek and lightweight design, it does not compromise on power, making it perfect for on-the-go professionals.
```

### Original quote/transcript 6

```text
Powered by powerful Intel/AMD processors, this laptop ensures seamless multitasking and high-performance computing. Its ultra-thin design with dimensions of 35 cm (13.78") in width, 24 cm (9.45") in depth, and 1.5 cm (0.59") in height, makes it highly portable without sacrificing power.
```

### Original quote/transcript 7

```text
The laptop boasts a crisp and vibrant display with a resolution tailored for immersive work or entertainment experiences. You can choose between SSD or HDD storage options to cater to your speed and capacity needs. It also offers various connectivity options, including USB-C, HDMI, and Thunderbolt, ensuring versatile compatibility.
```

### Original quote/transcript 8

```text
With different RAM configurations available, you can customize the laptop to suit your processing requirements. Additionally, you have the option to choose between Windows and MacOS operating systems, catering to your preferences.
```

### Original quote/transcript 9

```text
This laptop is built to last with its durable aluminum casing, providing a premium look and robust protection. The high-resolution display features an anti-glare coating, ensuring comfortable viewing in any lighting condition. It also incorporates cutting-edge cooling technology to ensure optimal performance during extended usage.
```

### Original quote/transcript 10

```text
Crafted with precision in state-of-the-art facilities in Japan, this laptop combines traditional craftsmanship with technological innovation. Experience the perfect blend of style, performance, and durability with our sleek Laptop Series.
```

### Original quote/transcript 11

```text
Introducing our sleek Laptop Series, a cutting-edge embodiment of design and functionality. Available in multiple colors and finishes, it seamlessly integrates with your personal or professional aesthetic. Crafted with premium materials, it offers durability and style. Engineered for superior performance, it’s perfect for professionals in any environment.
```

### Original quote/transcript 12

```text
47
```

### Original quote/transcript 13

```text
Introducing our sleek Laptop Series, crafted with premium materials for a durable and stylish appeal. Choose from various color options and finishes, including brushed aluminum, matte black, glossy white, or chrome accents. The laptop features a high-resolution display with anti-glare coating and cutting-edge cooling technology for optimal performance. Made with precision in state-of-the-art facilities in Japan.
```

### Original quote/transcript 14

```text
Introducing our sleek Laptop Series, crafted with premium materials for a durable and stylish appeal. The laptop is available in multiple colors and finishes, including brushed aluminum, matte black, glossy white, or chrome accents. It features a high-resolution display with anti-glare coating for comfortable viewing in any lighting condition. With powerful Intel/AMD processors and various storage and RAM options, this laptop is engineered for superior performance and productivity. It also offers cutting-edge cooling technology for optimal performance during extended usage. Choose our Laptop Series for a sophisticated and reliable device that seamlessly integrates with your personal or professional aesthetic.
```

### Original quote/transcript 15

```text
Product IDs: Laptop01, Laptop02
```

### Original quote/transcript 16

```text
<div>
 <h2>Product Description</h2>
 <p>
 Introducing our sleek Laptop Series, an embodiment of cutting-edge design and functionality for modern workspaces. Part of a sophisticated family of tech-inspired devices, including laptops, docking stations, accessories, and more. Available in multiple colors and finishes to seamlessly integrate with your personal or professional aesthetic. Crafted with premium materials for a durable and stylish appeal. Suitable for a variety of environments, whether it’s your home office or a corporate setting. Engineered for superior performance and productivity, making it an ideal choice for professionals.
 </p>
 <p>
 The laptop is available in various color options, allowing you to personalize it to your preference. Choose between different finishes, including brushed aluminum, matte black, glossy white, or chrome accents. Select from configurations with a standard display or a touchscreen option. Options for additional features such as a backlit keyboard, fingerprint sensor, or face recognition technology. Sleek and lightweight design for portability without compromising on power.
 </p>
 <p>
 Powerful Intel/AMD processors for seamless multitasking and high-performance computing. Crisp and vibrant display with a resolution tailored for immersive work or entertainment experiences. Choose between SSD or HDD storage options for speed and capacity tailored to your needs. Various connectivity options, including USB-C, HDMI, and Thunderbolt for versatile compatibility. Different RAM configurations to suit your processing requirements. Options for operating systems, catering to both Windows and MacOS preferences.
 </p>
 <p>
 Crafted with precision in state-of-the-art facilities in Japan, combining traditional craftsmanship with technological innovation.
 </p>
 
 <h2>Product Dimensions</h2>
 <table>
 <tr>
 <th>Name of Dimension</th>
 <th>Measurements (inches)</th>
 </tr>
 <tr>
 <td>Width</td>
 <td>13.78"</td>
 </tr>
 <tr>
 <td>Depth</td>
 <td>9.45"</td>
 </tr>
 <tr>
 <td>Height</td>
 <td>0.59"</td>
 </tr>
 </table>
</div>
```

### Original quote/transcript 17

```text
Product IDs: [To be provided based on the technical specifications]
```

### Original quote/transcript 18

```text
Product Description
```

### Original quote/transcript 19

```text
Introducing our sleek Laptop Series, an embodiment of cutting-edge design and functionality for modern workspaces. Part of a sophisticated family of tech-inspired devices, including laptops, docking stations, accessories, and more. Available in multiple colors and finishes to seamlessly integrate with your personal or professional aesthetic. Crafted with premium materials for a durable and stylish appeal. Suitable for a variety of environments, whether it’s your home office or a corporate setting. Engineered for superior performance and productivity, making it an ideal choice for professionals.
```

### Original quote/transcript 20

```text
The laptop is available in various color options, allowing you to personalize it to your preference. Choose between different finishes, including brushed aluminum, matte black, glossy white, or chrome accents. Select from configurations with a standard display or a touchscreen option. Options for additional features such as a backlit keyboard, fingerprint sensor, or face recognition technology. Sleek and lightweight design for portability without compromising on power.
```

### Original quote/transcript 21

```text
Powerful Intel/AMD processors for seamless multitasking and high-performance computing. Crisp and vibrant display with a resolution tailored for immersive work or entertainment experiences. Choose between SSD or HDD storage options for speed and capacity tailored to your needs. Various connectivity options, including USB-C, HDMI, and Thunderbolt for versatile compatibility. Different RAM configurations to suit your processing requirements. Options for operating systems, catering to both Windows and MacOS preferences.
```

### Original quote/transcript 22

```text
Crafted with precision in state-of-the-art facilities in Japan, combining traditional craftsmanship with technological innovation.
```